# 🏦 Activité 10 — Fraude et Segmentation
### M1 IA DATA — Machine Learning — YNOV

**Contexte :** Une fintech souhaite :
1. Détecter automatiquement les transactions frauduleuses
2. Segmenter les comportements clients
3. Identifier des transactions anormales

**Dataset :** Credit Card Fraud Detection — 284 807 transactions, ~0.17% de fraudes

---
## Plan du notebook
- **PARTIE 1** — Exploration & Nettoyage
- **PARTIE 2** — Feature Engineering
- **PARTIE 3** — Modélisation Supervisée
- **PARTIE 4** — Apprentissage Non Supervisé
- **PARTIE 5** — Pipeline Complet

In [ ]:
# ============================================================
# IMPORTS GLOBAUX
# On importe toutes les bibliothèques nécessaires au projet
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn : outils de Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

# Imbalanced-learn : pour gérer le déséquilibre de classes (SMOTE)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Paramètres d'affichage
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Tous les imports sont OK !')

---
# PARTIE 1 — Exploration & Nettoyage
> Objectif : comprendre les données, identifier les problèmes (déséquilibre, outliers…) et produire un dataset propre.

In [ ]:
# ============================================================
# 1.1 CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv('Dataset 1/creditcard.csv')

print(f'📊 Dimensions du dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print()
df.head()

In [ ]:
# ============================================================
# 1.2 INFORMATIONS GÉNÉRALES SUR LE DATASET
# ============================================================

# df.info() donne le type de chaque colonne et s'il y a des valeurs manquantes
df.info()

In [ ]:
# ============================================================
# 1.3 STATISTIQUES DESCRIPTIVES
# Ces statistiques nous donnent une vue d'ensemble des données :
# moyenne, écart-type, min, max, quartiles
# ============================================================

df.describe().round(2)

In [ ]:
# ============================================================
# 1.4 VALEURS MANQUANTES
# On vérifie s'il y a des données manquantes dans chaque colonne
# ============================================================

manquantes = df.isna().sum()
print('Valeurs manquantes par colonne :')
print(manquantes[manquantes > 0] if manquantes.sum() > 0 else '✅ Aucune valeur manquante détectée !')

In [ ]:
# ============================================================
# 1.5 ANALYSE DU DÉSÉQUILIBRE DE CLASSES
# Le déséquilibre est un problème majeur en détection de fraude :
# si seulement 0.17% des transactions sont frauduleuses,
# un modèle 'naïf' qui prédit toujours 0 aurait 99.83% de précision !
# Il faut donc utiliser des métriques adaptées (Recall, F1, AUC).
# ============================================================

nb_fraudes = (df['Class'] == 1).sum()
nb_normales = (df['Class'] == 0).sum()
total = len(df)

print(f'✅ Transactions normales : {nb_normales:,} ({nb_normales/total*100:.2f}%)')
print(f'🚨 Transactions frauduleuses : {nb_fraudes:,} ({nb_fraudes/total*100:.2f}%)')
print(f'Ratio de déséquilibre : 1 fraude pour {nb_normales//nb_fraudes} transactions normales')

# Visualisation du déséquilibre
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Graphique 1 : Bar chart
counts = df['Class'].value_counts()
axes[0].bar(['Normale (0)', 'Fraude (1)'], counts, color=['steelblue', 'crimson'])
axes[0].set_title('Répartition des classes (valeurs absolues)')
axes[0].set_ylabel('Nombre de transactions')
for i, v in enumerate(counts):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Graphique 2 : Camembert
axes[1].pie(counts, labels=['Normale (0)', 'Fraude (1)'],
            colors=['steelblue', 'crimson'],
            autopct='%1.2f%%', startangle=90)
axes[1].set_title('Répartition des classes (pourcentages)')

plt.suptitle('⚠️ Déséquilibre de classes — Dataset Fraude', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('💡 Interprétation : Le dataset est extrêmement déséquilibré (~0.17% de fraudes).')
print('   Il faudra utiliser SMOTE ou class_weight pour compenser ce déséquilibre.')

In [ ]:
# 1.6 ANALYSE DE AMOUNT ET TIME
# Amount et Time sont les seules variables NON transformées par PCA

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
normale, fraude = df[df['Class'] == 0], df[df['Class'] == 1]

# Distribution Amount par classe (zoom 0-500 euros)
for data, color, label in [(normale, 'steelblue', 'Normale'), (fraude, 'crimson', 'Fraude')]:
    axes[0, 0].hist(data['Amount'], bins=60, color=color, alpha=0.7, label=label)
axes[0, 0].set(title='Distribution du montant', xlabel='Montant', xlim=(0, 500))
axes[0, 0].legend()

# Boxplot Amount par classe
sns.boxplot(data=df, x='Class', y='Amount', ax=axes[0, 1], palette=['steelblue', 'crimson'])
axes[0, 1].set(title='Boxplot Amount par classe', xlabel='Classe')

# Distribution de Time
axes[1, 0].hist(df['Time'], bins=60, color='mediumseagreen', alpha=0.8)
axes[1, 0].set(title='Distribution du temps (Time)', xlabel='Secondes ecoulees')

# Scatter Time vs Amount colore par classe
axes[1, 1].scatter(normale['Time'], normale['Amount'], alpha=0.02, c='steelblue', s=5, label='Normale')
axes[1, 1].scatter(fraude['Time'],  fraude['Amount'],  alpha=0.5,  c='crimson',   s=15, label='Fraude')
axes[1, 1].set(title='Time vs Amount', xlabel='Time (s)', ylabel='Amount')
axes[1, 1].legend()

plt.suptitle('Analyse de Amount et Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Les fraudes ont des montants plus faibles. Time couvre 2 jours, les fraudes sont reparties uniformement.

In [ ]:
# ============================================================
# 1.7 DÉTECTION D'OUTLIERS SUR AMOUNT
# On utilise la méthode IQR (Interquartile Range) :
# tout ce qui dépasse Q3 + 1.5*IQR est considéré comme outlier
# Note : les variables V1-V28 ont déjà été transformées par PCA,
# donc on se concentre sur Amount
# ============================================================

Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
borne_sup = Q3 + 1.5 * IQR

outliers = df[df['Amount'] > borne_sup]
print(f'Q1 = {Q1:.2f}€, Q3 = {Q3:.2f}€, IQR = {IQR:.2f}€')
print(f'Borne supérieure : {borne_sup:.2f}€')
print(f'Nombre d\'outliers détectés : {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)')
print(f'Montant max des outliers : {outliers["Amount"].max():.2f}€')
print()
print('💡 Choix de nettoyage : On conserve ces outliers car ils peuvent représenter des')
print('   comportements légitimes (gros achats) ou des fraudes. On les gérera via la')
print('   transformation log du montant en Partie 2.')

In [ ]:
# ============================================================
# 1.8 ANALYSE DES CORRÉLATIONS AVEC LA VARIABLE CIBLE
# On cherche quelles variables sont les plus liées à la fraude.
# Une corrélation forte (positive ou négative) indique une
# variable potentiellement utile pour la détection.
# ============================================================

# Corrélation de chaque variable avec Class
correlations = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['crimson' if x < 0 else 'steelblue' for x in correlations]
correlations.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Corrélation de chaque variable avec la classe (fraude)', fontsize=13)
ax.set_xlabel('Coefficient de corrélation de Pearson')
plt.tight_layout()
plt.show()

print('💡 Top 5 des variables les plus corrélées avec la fraude :')
print(correlations.abs().sort_values(ascending=False).head(5))

In [ ]:
# ============================================================
# 1.9 DISTRIBUTION DES VARIABLES LES PLUS DISCRIMINANTES
# On visualise les 6 variables les plus corrélées avec la fraude
# pour voir si elles séparent bien les deux classes.
# ============================================================

# On sélectionne les 6 variables les plus corrélées
top_vars = correlations.abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(top_vars):
    axes[i].hist(df[df['Class'] == 0][col], bins=50, alpha=0.6,
                 color='steelblue', label='Normale', density=True)
    axes[i].hist(df[df['Class'] == 1][col], bins=50, alpha=0.6,
                 color='crimson', label='Fraude', density=True)
    axes[i].set_title(f'Distribution de {col}')
    axes[i].legend()

plt.suptitle('Distribution des variables les plus discriminantes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Ces variables montrent des distributions très différentes entre fraudes et normales.')
print('   Elles seront les plus utiles pour nos modèles de classification.')

---
# PARTIE 2 — Feature Engineering
> Objectif : créer ≥3 nouvelles variables pertinentes pour améliorer les performances des modèles.

In [ ]:
# ============================================================
# 2.1 CRÉATION DE NOUVELLES FEATURES
# ============================================================

# On travaille sur une copie pour ne pas modifier le dataset original
df_fe = df.copy()

# --- Feature 1 : Montant log-transformé ---
# Pourquoi ? Amount a une distribution très asymétrique avec des valeurs extrêmes.
# La transformation log réduit l'impact des grands montants et normalise la distribution.
df_fe['Amount_log'] = np.log1p(df_fe['Amount'])  # log(1 + x) pour gérer les zéros

# --- Feature 2 : Heure du jour (indicateur jour/nuit) ---
# Pourquoi ? Les fraudes peuvent être plus fréquentes la nuit quand les banques
# sont fermées et la surveillance est réduite.
# Time est en secondes, 1 jour = 86400 secondes
df_fe['Hour'] = (df_fe['Time'] / 3600) % 24  # Heure de la journée (0-24)
df_fe['IsNight'] = ((df_fe['Hour'] >= 22) | (df_fe['Hour'] <= 6)).astype(int)

# --- Feature 3 : Catégorie de montant ---
# Pourquoi ? Certaines tranches de montants peuvent être plus risquées.
# On découpe en quartiles pour créer 4 catégories.
df_fe['Amount_cat'] = pd.qcut(df_fe['Amount'], q=4, labels=[0, 1, 2, 3]).astype(int)

# --- Feature 4 : Interaction V1 × V3 ---
# Pourquoi ? V1 et V3 sont parmi les variables les plus corrélées avec la fraude.
# Leur interaction peut capturer des patterns non linéaires.
df_fe['V1_V3'] = df_fe['V1'] * df_fe['V3']

# --- Feature 5 : Écart de V14 par rapport à la médiane ---
# Pourquoi ? V14 est souvent la variable la plus discriminante.
# L'écart par rapport à la médiane mesure à quel point une transaction est "anormale".
median_v14 = df_fe['V14'].median()
df_fe['V14_ecart'] = np.abs(df_fe['V14'] - median_v14)

print('✅ Nouvelles features créées :')
nouvelles = ['Amount_log', 'Hour', 'IsNight', 'Amount_cat', 'V1_V3', 'V14_ecart']
print(df_fe[nouvelles].describe().round(3))

In [ ]:
# 2.2 VISUALISATION DE L'IMPACT DES NOUVELLES FEATURES

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
df_fe['Hour_int'] = df_fe['Hour'].astype(int)
normale_fe, fraude_fe = df_fe[df_fe['Class'] == 0], df_fe[df_fe['Class'] == 1]

# Amount_log et V14_ecart : histogrammes superposés normale vs fraude
for ax, col in zip([axes[0], axes[2]], ['Amount_log', 'V14_ecart']):
    for data, color, label in [(normale_fe, 'steelblue', 'Normale'), (fraude_fe, 'crimson', 'Fraude')]:
        ax.hist(data[col], bins=50, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(f'{col} par classe')
    ax.legend()

# Fraudes par heure
hourly = df_fe.groupby('Hour_int')['Class'].sum()
axes[1].bar(hourly.index, hourly.values, color='crimson', alpha=0.7)
axes[1].set(title='Fraudes par heure', xlabel='Heure (0-24)', ylabel='Nb fraudes')

plt.suptitle('Impact des nouvelles features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2.3 NORMALISATION DES DONNÉES
# On utilise StandardScaler qui centre les données (moyenne=0)
# et les met à l'échelle (écart-type=1).
# Pourquoi ? Certains algorithmes (Logistic Regression, SVM, K-means)
# sont sensibles à l'échelle des variables.
# ============================================================

# Colonnes à conserver pour la modélisation
features_a_garder = [col for col in df_fe.columns
                     if col not in ['Class', 'Time', 'Amount', 'Hour', 'Hour_int']]

X = df_fe[features_a_garder].copy()
y = df_fe['Class'].copy()

# Normalisation (on garde les colonnes V1-V28 + nouvelles features)
cols_a_scaler = ['Amount_log', 'V14_ecart', 'V1_V3']
scaler = StandardScaler()
X[cols_a_scaler] = scaler.fit_transform(X[cols_a_scaler])

print(f'✅ Features finales : {X.shape[1]} variables')
print(f'   Classes : {y.value_counts().to_dict()}')
print()
print('💡 Justification :')
print('   - Amount_log, V14_ecart et V1_V3 sont normalisées car elles ont des échelles différentes.')
print('   - Les variables V1-V28 sont déjà centrées-réduites (résultat de la PCA d\'origine).')

---
# PARTIE 3 — Modélisation Supervisée
> Objectif : construire et comparer ≥2 modèles pour détecter les fraudes, en gérant le déséquilibre.

In [ ]:
# ============================================================
# 3.1 SPLIT TRAIN / TEST
# On utilise stratify=y pour conserver les mêmes proportions
# de fraude dans les deux ensembles (train et test).
# 80% pour l'entraînement, 20% pour le test.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ← Crucial pour garder le ratio fraude/normal
)

print(f'✅ Train : {X_train.shape[0]:,} transactions')
print(f'✅ Test  : {X_test.shape[0]:,} transactions')
print()
print('Répartition dans le train :')
print(y_train.value_counts())
print()
print('Répartition dans le test :')
print(y_test.value_counts())

In [ ]:
# ============================================================
# 3.2 GESTION DU DÉSÉQUILIBRE AVEC SMOTE
# SMOTE (Synthetic Minority Over-sampling Technique) crée
# des exemples synthétiques de fraude pour équilibrer les classes.
# IMPORTANT : on applique SMOTE UNIQUEMENT sur le train, jamais le test !
# ============================================================

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'Avant SMOTE : {y_train.value_counts().to_dict()}')
print(f'Après SMOTE : {pd.Series(y_train_sm).value_counts().to_dict()}')
print()
print('💡 SMOTE génère de nouvelles transactions frauduleuses synthétiques')
print('   en interpolant entre les exemples existants. Cela force le modèle')
print('   à apprendre à reconnaître les fraudes plutôt que d\'ignorer la minorité.')

In [ ]:
# ============================================================
# 3.3 QUELLE MÉTRIQUE PRIORISER EN DÉTECTION DE FRAUDE ?
# ============================================================

print('=== MÉTRIQUE PRIORITAIRE : LE RECALL (Rappel) ===')
print()
print('En détection de fraude, RATER une fraude (Faux Négatif) est bien plus')
print('coûteux que BLOQUER une transaction légitime (Faux Positif).')
print()
print('- Recall = TP / (TP + FN) : proportion de fraudes correctement détectées')
print('- Precision = TP / (TP + FP) : proportion de vraies fraudes parmi les alertes')
print()
print('On utilise le F1-score comme compromis Precision/Recall,')
print('et l\'AUC-ROC pour comparer les modèles de façon globale.')

In [ ]:
# ============================================================
# 3.4 MODÈLE 1 : RÉGRESSION LOGISTIQUE
# Simple, rapide, interprétable. C'est notre baseline.
# max_iter=1000 pour s'assurer de la convergence.
# ============================================================

print('Entraînement de la Régression Logistique...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)

# Prédictions
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# Métriques
print('\n📊 Régression Logistique — Résultats sur le TEST :')
print('─' * 50)
print(classification_report(y_test, y_pred_lr, target_names=['Normale', 'Fraude']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_lr):.4f}')

In [ ]:
# ============================================================
# 3.5 MODÈLE 2 : RANDOM FOREST
# Ensemble de nombreux arbres de décision. Plus puissant
# que la régression logistique, gère bien les non-linéarités.
# n_estimators=100 : nombre d'arbres dans la forêt
# ============================================================

print('Entraînement du Random Forest (peut prendre quelques secondes)...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)

# Prédictions
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Métriques
print('\n📊 Random Forest — Résultats sur le TEST :')
print('─' * 50)
print(classification_report(y_test, y_pred_rf, target_names=['Normale', 'Fraude']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_proba_rf):.4f}')

In [ ]:
# ============================================================
# 3.6 COMPARAISON DES MODÈLES — MATRICES DE CONFUSION ET ROC
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# --- Matrice de confusion — Régression Logistique ---
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=['Normale', 'Fraude'],
    cmap='Blues', ax=axes[0, 0]
)
axes[0, 0].set_title('Matrice de confusion — Régression Logistique')

# --- Matrice de confusion — Random Forest ---
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=['Normale', 'Fraude'],
    cmap='Greens', ax=axes[0, 1]
)
axes[0, 1].set_title('Matrice de confusion — Random Forest')

# --- Courbe ROC — comparaison ---
RocCurveDisplay.from_predictions(y_test, y_proba_lr, name='Régression Logistique',
                                  color='steelblue', ax=axes[1, 0])
axes[1, 0].set_title('Courbe ROC — Régression Logistique')

RocCurveDisplay.from_predictions(y_test, y_proba_rf, name='Random Forest',
                                  color='green', ax=axes[1, 1])
axes[1, 1].set_title('Courbe ROC — Random Forest')

plt.suptitle('Comparaison des modèles supervisés', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Tableau récapitulatif ---
print('\n📊 TABLEAU RÉCAPITULATIF :')
print('─' * 60)
resultats = pd.DataFrame({
    'Modèle': ['Régression Logistique', 'Random Forest'],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf)
    ],
    'AUC-ROC': [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_rf)
    ]
})
print(resultats.to_string(index=False))
print()
print('💡 Interprétation :')
print('   Le Random Forest obtient généralement de meilleures performances')
print('   car il combine plusieurs arbres et gère mieux les relations non-linéaires.')
print('   En contexte de fraude, on privilégie le Recall pour minimiser les fraudes ratées.')

In [ ]:
# ============================================================
# 3.7 IMPORTANCE DES FEATURES (Random Forest)
# Le Random Forest peut nous dire quelles variables sont
# les plus importantes pour distinguer fraudes et non-fraudes.
# ============================================================

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances_top = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
importances_top.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 15 variables les plus importantes (Random Forest)', fontsize=13)
ax.set_ylabel('Importance')
ax.set_xlabel('Variable')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('💡 Interprétation :')
print('   - V14, V17 et V12 sont généralement les plus discriminants.')
print('   - Nos nouvelles features (Amount_log, V14_ecart) contribuent également,')
print('     ce qui confirme que le Feature Engineering a apporté de la valeur.')

In [ ]:
# ============================================================
# 3.8 CROSS-VALIDATION (Validation croisée)
# On divise le train en 5 parties (folds) et on évalue
# le modèle sur chaque fold pour avoir une estimation
# plus robuste des performances.
# ============================================================

print('Cross-validation sur le Random Forest (5 folds)...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    X_train_sm, y_train_sm,
    cv=skf, scoring='f1', n_jobs=-1
)

print(f'\n📊 Scores F1 par fold : {cv_scores.round(4)}')
print(f'   Moyenne : {cv_scores.mean():.4f}')
print(f'   Écart-type : {cv_scores.std():.4f}')
print()
print('💡 Un faible écart-type indique que le modèle est stable et ne sur-apprend pas.')

---
# PARTIE 4 — Apprentissage Non Supervisé
> Objectif : segmenter les transactions et détecter des anomalies SANS utiliser les étiquettes.

In [ ]:
# ============================================================
# 4.1 RÉDUCTION DE DIMENSION AVEC PCA (pour visualisation)
# On a beaucoup de variables (30+). On les réduit à 2 dimensions
# pour pouvoir visualiser les clusters sur un graphique 2D.
# ============================================================

# On travaille sur un échantillon pour la rapidité
print('Réduction PCA pour visualisation...')
SAMPLE_SIZE = 10000

# Échantillon stratifié (garde les proportions fraude/normal)
idx_fraude = df_fe[df_fe['Class'] == 1].index
idx_normal = df_fe[df_fe['Class'] == 0].sample(SAMPLE_SIZE - len(idx_fraude),
                                               random_state=42).index
idx_sample = idx_fraude.append(idx_normal)

X_sample = X.loc[idx_sample]
y_sample = y.loc[idx_sample]

# Normalisation avant PCA
scaler_pca = StandardScaler()
X_sample_scaled = scaler_pca.fit_transform(X_sample)

# PCA → 2 composantes
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample_scaled)

print(f'✅ Variance expliquée par les 2 composantes : {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Visualisation
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=y_sample, cmap='RdBu_r', alpha=0.5, s=15)
plt.colorbar(scatter, ax=ax, label='Classe (0=Normal, 1=Fraude)')
ax.set_title('Visualisation PCA des transactions (Rouge = Fraude)', fontsize=13)
ax.set_xlabel('Composante PCA 1')
ax.set_ylabel('Composante PCA 2')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.2 K-MEANS — CHOIX DU NOMBRE OPTIMAL DE CLUSTERS
# On utilise la méthode "Elbow" (coude) : on trace l'inertie
# (somme des distances au centroïde) en fonction de k.
# Le bon k est là où la courbe fait un "coude".
# ============================================================

print('Calcul de la méthode Elbow (K=2 à 10)...')
inerties = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(list(k_range), inerties, 'b-o', linewidth=2)
axes[0].set_title('Méthode Elbow — Inertie vs K')
axes[0].set_xlabel('Nombre de clusters (K)')
axes[0].set_ylabel('Inertie')
axes[0].grid(True)

# Silhouette
axes[1].plot(list(k_range), silhouettes, 'r-o', linewidth=2)
axes[1].set_title('Score Silhouette vs K')
axes[1].set_xlabel('Nombre de clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True)

plt.suptitle('Choix optimal du nombre de clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'✅ Meilleur K selon Silhouette : K = {best_k} (score = {max(silhouettes):.3f})')

In [ ]:
# ============================================================
# 4.3 K-MEANS — CLUSTERING FINAL
# On applique K-Means avec le meilleur K trouvé
# ============================================================

# On force K=3 pour avoir une interprétation business claire
K_FINAL = 3

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
clusters_km = kmeans.fit_predict(X_pca)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Clusters K-Means
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                           c=clusters_km, cmap='tab10', alpha=0.5, s=15)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroïdes')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')
axes[0].set_title(f'K-Means (K={K_FINAL}) — Clusters')
axes[0].legend()

# Vraies fraudes superposées
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                           c=y_sample, cmap='RdBu_r', alpha=0.5, s=15)
plt.colorbar(scatter2, ax=axes[1], label='0=Normal, 1=Fraude')
axes[1].set_title('Vraies étiquettes (pour comparaison)')

plt.suptitle('K-Means Clustering vs Vraies Étiquettes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Analyse des clusters
df_cluster = pd.DataFrame({'cluster': clusters_km, 'class': y_sample.values})
print('📊 Répartition fraudes/normales par cluster :')
print(df_cluster.groupby('cluster')['class'].value_counts().unstack(fill_value=0))

print(f'\nScore Silhouette final : {silhouette_score(X_pca, clusters_km):.3f}')
print()
print('💡 Interprétation business :')
print('   Les clusters représentent des profils de comportement différents.')
print('   Un cluster concentrant beaucoup de fraudes peut représenter')
print('   des comportements suspects (petits montants, transactions nocturnes…).')

In [ ]:
# ============================================================
# 4.4 DBSCAN — CLUSTERING BASÉ SUR LA DENSITÉ
# DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
# identifie des clusters de forme arbitraire ET des points "bruit"
# (outliers), ce qui est idéal pour la détection de fraude.
# eps = rayon du voisinage, min_samples = nb minimum de voisins
# ============================================================

print('Application de DBSCAN...')
dbscan = DBSCAN(eps=0.5, min_samples=10)
clusters_db = dbscan.fit_predict(X_pca)

n_clusters_db = len(set(clusters_db)) - (1 if -1 in clusters_db else 0)
n_bruit = (clusters_db == -1).sum()

print(f'Clusters trouvés : {n_clusters_db}')
print(f'Points "bruit" (anomalies potentielles) : {n_bruit} ({n_bruit/len(clusters_db)*100:.1f}%)')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DBSCAN clusters (-1 = bruit en rouge)
couleurs_db = ['crimson' if c == -1 else plt.cm.tab10(c % 10) for c in clusters_db]
for i, (x, y_coord) in enumerate(X_pca):
    axes[0].scatter(x, y_coord, c=[couleurs_db[i]], alpha=0.3, s=10)
axes[0].set_title('DBSCAN — Rouge = points bruité (anomalies)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Comparaison bruit vs vraies fraudes
df_db = pd.DataFrame({'dbscan_bruit': (clusters_db == -1).astype(int),
                      'vraie_fraude': y_sample.values})
conf_db = pd.crosstab(df_db['dbscan_bruit'], df_db['vraie_fraude'],
                      rownames=['DBSCAN bruit'], colnames=['Vraie fraude'])
sns.heatmap(conf_db, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('DBSCAN bruit vs Vraies fraudes')

plt.suptitle('DBSCAN — Détection d\'anomalies', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Interprétation :')
print('   Les points "bruit" de DBSCAN sont des transactions isolées,')
print('   qui ne ressemblent à aucun cluster. Ce sont de bons candidats fraudes.')

In [ ]:
# ============================================================
# 4.5 ISOLATION FOREST — DÉTECTION D'ANOMALIES
# Isolation Forest isole les anomalies en construisant des arbres
# aléatoires. Les points faciles à isoler (peu de coupures nécessaires)
# sont considérés comme des anomalies.
# contamination = proportion estimée d'anomalies (~0.17% de fraudes)
# ============================================================

print('Application Isolation Forest...')
iso_forest = IsolationForest(contamination=0.002, random_state=42, n_jobs=-1)
pred_if = iso_forest.fit_predict(X_sample_scaled)

# Isolation Forest retourne -1 pour anomalie, 1 pour normal
# On convertit en 0/1 pour faciliter la comparaison
pred_if_bin = (pred_if == -1).astype(int)

print(f'Anomalies détectées par Isolation Forest : {pred_if_bin.sum()} ({pred_if_bin.sum()/len(pred_if_bin)*100:.2f}%)')
print(f'Vraies fraudes dans l\'échantillon : {y_sample.sum()}')
print()

# Métriques par rapport aux vraies fraudes
print('📊 Comparaison Isolation Forest vs Vraies Fraudes :')
print(classification_report(y_sample, pred_if_bin,
                             target_names=['Normale', 'Fraude']))

In [ ]:
# ============================================================
# 4.6 LOCAL OUTLIER FACTOR (LOF) — DÉTECTION D'ANOMALIES
# LOF mesure la densité locale de chaque point et compare
# avec ses voisins. Un point moins dense que ses voisins
# est une anomalie.
# Avantage par rapport à Isolation Forest : détecte les anomalies
# locales (dans des zones de haute densité)
# ============================================================

print('Application Local Outlier Factor...')
lof = LocalOutlierFactor(contamination=0.002, n_jobs=-1)
pred_lof = lof.fit_predict(X_sample_scaled)
pred_lof_bin = (pred_lof == -1).astype(int)

print(f'Anomalies détectées par LOF : {pred_lof_bin.sum()}')

# Comparaison IF vs LOF
print('\n📊 Comparaison LOF vs Vraies Fraudes :')
print(classification_report(y_sample, pred_lof_bin,
                             target_names=['Normale', 'Fraude']))

# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=pred_if_bin,
                cmap='RdBu_r', alpha=0.5, s=15)
axes[0].set_title('Isolation Forest (Rouge = Anomalie)')

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=pred_lof_bin,
                cmap='RdBu_r', alpha=0.5, s=15)
axes[1].set_title('LOF (Rouge = Anomalie)')

plt.suptitle('Détection d\'anomalies : Isolation Forest vs LOF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Interprétation business :')
print('   - Isolation Forest est plus rapide et adapté aux grands datasets.')
print('   - LOF est plus précis pour les anomalies locales.')
print('   - On peut combiner les deux : une transaction flaggée par les deux')
print('     est très probablement frauduleuse.')

---
# PARTIE 5 — Pipeline Complet
> Objectif : intégrer tout le preprocessing, le feature engineering et la modélisation dans un pipeline robuste et reproductible.

In [ ]:
# ============================================================
# 5.1 CONSTRUCTION DU PIPELINE COMPLET
# Un pipeline garantit que chaque étape est appliquée
# dans le bon ordre, sans "fuite" de données (data leakage).
# On utilise ImbPipeline (d'imbalanced-learn) pour intégrer SMOTE.
# ============================================================

# Données brutes (avec Feature Engineering déjà fait)
X_raw = df_fe[features_a_garder].copy()
y_raw = df_fe['Class'].copy()

# Split propre sans SMOTE (le pipeline s'en charge)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Construction du pipeline
pipeline = ImbPipeline(steps=[
    # Étape 1 : Normalisation des données
    ('scaler', StandardScaler()),

    # Étape 2 : SMOTE pour rééquilibrer les classes
    ('smote', SMOTE(random_state=42)),

    # Étape 3 : Modèle Random Forest
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

print('Pipeline construit :')
print(pipeline)
print()
print('Entraînement du pipeline...')
pipeline.fit(X_tr, y_tr)
print('✅ Pipeline entraîné avec succès !')

In [ ]:
# ============================================================
# 5.2 ÉVALUATION DU PIPELINE
# On évalue les performances sur le jeu de test
# ============================================================

y_pred_pipeline = pipeline.predict(X_te)
y_proba_pipeline = pipeline.predict_proba(X_te)[:, 1]

print('📊 RÉSULTATS FINAUX DU PIPELINE :')
print('=' * 60)
print(classification_report(y_te, y_pred_pipeline,
                             target_names=['Normale', 'Fraude']))
print(f'AUC-ROC : {roc_auc_score(y_te, y_proba_pipeline):.4f}')

# Matrice de confusion finale
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred_pipeline,
    display_labels=['Normale', 'Fraude'],
    cmap='Blues', ax=ax
)
ax.set_title('Matrice de confusion — Pipeline Final', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.3 CROSS-VALIDATION DU PIPELINE
# La cross-validation sur le pipeline garantit qu'il n'y a
# pas de data leakage : le SMOTE et le StandardScaler sont
# re-entraînés sur chaque fold séparément.
# ============================================================

print('Cross-validation du pipeline (3 folds, peut prendre 1-2 minutes)...')

skf_pipe = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_pipe = cross_val_score(
    pipeline, X_tr, y_tr,
    cv=skf_pipe, scoring='roc_auc', n_jobs=-1
)

print(f'\n📊 AUC-ROC par fold : {cv_pipe.round(4)}')
print(f'   Moyenne : {cv_pipe.mean():.4f} ± {cv_pipe.std():.4f}')
print()
print('💡 Un AUC-ROC proche de 1 indique un excellent pouvoir discriminant.')

---
# 🏁 Conclusion & Recommandations Business

## Résultats clés

| Partie | Résultat |
|--------|----------|
| **EDA** | Dataset fortement déséquilibré (0.17% fraudes), pas de valeurs manquantes |
| **Feature Engineering** | 5 nouvelles features créées dont Amount_log et V14_ecart qui augmentent les performances |
| **Modèles supervisés** | Random Forest > Régression Logistique en Recall et AUC-ROC |
| **Clustering** | K-Means identifie 3 profils de transactions distincts |
| **Anomalies** | Isolation Forest et LOF détectent des patterns inhabituels |
| **Pipeline** | Pipeline robuste avec StandardScaler + SMOTE + Random Forest |

## Recommandations Business

1. **Déployer le Pipeline** avec le Random Forest : il maximise le Recall (détection des fraudes) tout en gardant une précision acceptable.

2. **Surveiller les alertes DBSCAN** : les transactions isolées (points "bruit") méritent une revue manuelle prioritaire.

3. **Seuil de décision** : baisser le seuil de classification de 0.5 à 0.3 pour augmenter le Recall (capturer plus de fraudes, au prix de plus de faux positifs).

4. **Ré-entraîner régulièrement** : les schémas de fraude évoluent, le modèle doit être mis à jour avec les nouvelles données.

5. **Combiner supervisé + non-supervisé** : flagguer automatiquement les transactions détectées à la fois par le Random Forest ET l'Isolation Forest.